# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their fields. Entities are referenced by their `@id`.

Let's enumerate all available record sets, fields, and columns. This helps identify the correct `@id`s for use in later steps.

In [ ]:
# Enumerate record sets, fields, and columns by their `@id`.
recordset_ids = []
print("Available Record Sets (with @id):\n---------------------------")
for rs in metadata.record_sets:
    print(f"RecordSet name: {rs.name}, @id: {rs.id}")
    recordset_ids.append(rs.id)
    print("Fields:")
    for field in rs.fields:
        print(f"  - {field.name} (field @id: {field.id}, dtype: {field.data_type})")
        # If field is a column with source, show the original column id
        if hasattr(field, 'columns') and field.columns:
            for col in field.columns:
                print(f"    - column @id: {col.id}, name: {col.name}, dtype: {col.data_type}")
    print('-' * 40)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s above.

We will extract all record sets, storing them in a dictionary of DataFrames, using their `@id` as the key.

In [ ]:
# Extract data from each record set into DataFrames
dataframes = {}

for record_set_id in recordset_ids:
    print(f"Loading records for RecordSet @id = {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")

# For this notebook, let's pick the first available record set for demonstration:
selected_recordset_id = recordset_ids[0] if recordset_ids else None
if selected_recordset_id:
    print(f'\nPreview of DataFrame for RecordSet @id: {selected_recordset_id}')
    display(dataframes[selected_recordset_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

Here we demonstrate filtering on a numeric field (if present), normalizing the field, and grouping by a relevant categorical field.

In [ ]:
import numpy as np
# Try to select a numeric field dynamically from the first record set
numeric_field = None
group_field = None

# Find numeric columns based on dtypes in DataFrame
if selected_recordset_id:
    df = dataframes[selected_recordset_id]

    for col in df.columns:
        try:
            # Attempt to convert column to numeric
            s = pd.to_numeric(df[col], errors='coerce')
            if s.notnull().sum() > 0 and (s.dtype == float or s.dtype == int or np.issubdtype(s.dtype, np.number)):
                numeric_field = col
                break
        except Exception:
            continue

    print(f"Selected numeric field: {numeric_field}")
    # Find grouping categorical field if possible (not the numeric field)
    candidate_cols = [c for c in df.columns if c != numeric_field]
    for col in candidate_cols:
        if df[col].dtype == object and df[col].nunique() < 15:
            group_field = col
            break

    print(f"Grouping by field: {group_field}")

    # Now, perform simple filtering, normalization, and grouping as EDA
    if numeric_field is not None:
        df_num = pd.to_numeric(df[numeric_field], errors='coerce')

        threshold = df_num.mean() if not np.isnan(df_num.mean()) else 0
        print(f"Filtering records where {numeric_field} > {threshold:.2f}")
        filtered_df = df[df_num > threshold].copy()
        print(filtered_df[[numeric_field]].head())

        # Normalization
        mean, std = df_num.mean(), df_num.std()
        if std and not np.isnan(std):
            filtered_df[f"{numeric_field}_normalized"] = (pd.to_numeric(filtered_df[numeric_field], errors='coerce') - mean) / std
            print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Grouping
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print("Grouped data:")
            print(grouped_df)
    else:
        print("No numeric field found for EDA in the selected record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll use matplotlib and seaborn if installed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_recordset_id and numeric_field:
    df = dataframes[selected_recordset_id]
    try:
        # Histogram for the numeric field
        plt.figure(figsize=(8,4))
        sns.histplot(pd.to_numeric(df[numeric_field], errors='coerce').dropna(), kde=True)
        plt.title(f'Distribution of {numeric_field} in RecordSet @id: {selected_recordset_id}')
        plt.xlabel(numeric_field)
        plt.show()

        # If grouping is available, show boxplot
        if group_field:
            plt.figure(figsize=(10,5))
            sns.boxplot(x=df[group_field], y=pd.to_numeric(df[numeric_field], errors='coerce'))
            plt.title(f'{numeric_field} by {group_field}')
            plt.xlabel(group_field)
            plt.ylabel(numeric_field)
            plt.show()
    except Exception as e:
        print(f"Visualization failed: {e}")
else:
    print("No suitable numeric field or record set for visualization.")

## 6. Conclusion
In this notebook, we:
- Explored the Croissant dataset using the `mlcroissant` library
- Reviewed all available record sets and fields by their `@id`
- Loaded records into pandas DataFrames for flexible analysis
- Performed simple EDA by filtering, normalizing, and grouping numeric fields
- Visualized numeric field distributions

This workflow helps bootstrap further analysis as record set and field `@id`s are retained and referenced throughout. For detailed domain analysis or advanced modeling, continue exploring with record set and field `@id`s defined above.